# Task 3: Federal Reserve Estimate Replication and Feature Preprocessing for 2025

In [5]:
from google.colab import files
import pandas as pd
import numpy as np

In [6]:
uploaded = files.upload()

Saving public2025_clean.csv to public2025_clean.csv


In [7]:
df = pd.read_csv(
    "/content/public2025_clean.csv",
    low_memory=False
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (12934, 815)


,shedid,duration,weight,weight_pop,panel_weight,panel_weight_pop,xlaptop,L0_a,L0_b,L0_c,...,E12_g_iflag,CH2A_iflag,race_5cat,inc_4cat_50k,educ_4cat,pay_casheqv,atleast_okay,control,malefemale,year
0,202304484,1922,0.6467,13225.3400,NaN,NaN,No,Yes,No,No,...,0,0,White,"$50,000–$99,999",Some college/technical or associates degree,Yes,Yes,Public,Male,2025
1,202204577,189270,0.9687,19810.3406,0.9357,56011.816,No,No,No,No,...,0,0,White,"$100,000 or more",Bachelor's degree or more,Yes,Yes,Public,Female,2025
2,202301342,4240,1.0129,20715.6590,NaN,NaN,No,No,No,Yes,...,0,0,White,"$50,000–$99,999",Less than a high school degree,No,No,__NOT_ASKED__,Female,2025
3,202500830,591,1.0278,21020.6151,NaN,NaN,No,Yes,No,No,...,0,0,White,"$50,000–$99,999",High school degree or GED,Yes,Yes,__NOT_ASKED__,Male,2025
4,202504363,2508,0.7688,15722.4241,NaN,NaN,No,Yes,No,No,...,0,0,White,"$50,000–$99,999",Some college/technical or associates degree,Yes,Yes,Public,Female,2025


In [9]:
required_columns = [
    "shedid",
    "weight_pop",
    "pay_casheqv"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Data validation complete and all required columns are present.")
print("Number of duplicate respondent IDs:", df["shedid"].duplicated().sum())
print("Number of missing population weights:", df["weight_pop"].isna().sum())
print("Number of missing target values:", df["pay_casheqv"].isna().sum())

print("\nTarget response counts:")
print(df["pay_casheqv"].value_counts(dropna=False))

Data validation complete and all required columns are present.
Number of duplicate respondent IDs: 0
Number of missing population weights: 0
Number of missing target values: 0

Target response counts:
pay_casheqv
Yes    8398
No     4536
Name: count, dtype: int64


After uploading, the file will be available in the `/content/` directory. You can then use its name (e.g., `public2025_clean.csv`) directly in your `pd.read_csv()` function.

Replication Standard

In [10]:
fed_estimate = 63.0
tolerance = 0.5

can_cover_expense = (
    df["pay_casheqv"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
    .astype(int)
)

weighted_result = (
    (
        can_cover_expense * df["weight_pop"]
    ).sum()
    / df["weight_pop"].sum()
) * 100

regular_result = can_cover_expense.mean() * 100

estimate_gap = abs(
    weighted_result - fed_estimate
)

passed_check = estimate_gap <= tolerance

print(f"Federal Reserve result: {fed_estimate:.2f}%")
print(f"Our weighted result: {weighted_result:.2f}%")
print(f"Our result without weights(regular result): {regular_result:.2f}%")
print(f"Distance from published result: {estimate_gap:.2f} points")
print(f"Maximum allowed distance: {tolerance:.2f} points")
print(f"Result is within tolerance: {passed_check}")

Federal Reserve result: 63.00%
Our weighted result: 63.14%
Our result without weights(regular result): 64.93%
Distance from published result: 0.14 points
Maximum allowed distance: 0.50 points
Result is within tolerance: True


Replication Result

In [11]:
financial_fragility = df["pay_casheqv"].map({
    "No": 1,
    "Yes": 0
})

if financial_fragility.isna().any():
    raise ValueError(
        "The target contains a missing or unexpected response."
    )

population_weights = df["weight_pop"].copy()

print("Financial-fragility target counts:")
print(financial_fragility.value_counts().sort_index())

print("\nNumber of population weights:", len(population_weights))
print("Number of missing population weights:", population_weights.isna().sum())

Financial-fragility target counts:
pay_casheqv
0    8398
1    4536
Name: count, dtype: int64

Number of population weights: 12934
Number of missing population weights: 0


Feature Selection

In [13]:
imputation_flag_columns = [
    name for name in df.columns
    if name.endswith("_iflag")
]

emergency_answer_columns = [
    "EF3_a",
    "EF3_b",
    "EF3_c",
    "EF3_d",
    "EF3_e",
    "EF3_f",
    "EF3_g",
    "EF3_h"
]

non_feature_columns = [
    "shedid",
    "duration",
    "weight",
    "weight_pop",
    "panel_weight",
    "panel_weight_pop",
    "pay_casheqv"
]

excluded_columns = (
    imputation_flag_columns
    + emergency_answer_columns
    + non_feature_columns
)

print("Number of _iflag columns excluded:", len(imputation_flag_columns))
print("Number of EF3 columns excluded:", len(emergency_answer_columns))
print("Other non-feature columns excluded:", len(non_feature_columns))
print("Total columns excluded:", len(excluded_columns))

Number of _iflag columns excluded: 357
Number of EF3 columns excluded: 8
Other non-feature columns excluded: 7
Total columns excluded: 372


In [14]:
features = df.drop(
    columns=excluded_columns,
    errors="ignore"
).copy()

print("Original dataset shape:", df.shape)
print("Feature matrix shape:", features.shape)

Original dataset shape: (12934, 815)
Feature matrix shape: (12934, 443)


In [15]:
remaining_iflags = [
    name for name in features.columns
    if name.endswith("_iflag")
]

remaining_ef3_columns = [
    name for name in emergency_answer_columns
    if name in features.columns
]

print("Remaining _iflag variables:", len(remaining_iflags))
print("Remaining EF3 columns:", len(remaining_ef3_columns))
print("Target still in features:", "pay_casheqv" in features.columns)
print("Population weight still in features:", "weight_pop" in features.columns)

Remaining _iflag variables: 0
Remaining EF3 columns: 0
Target still in features: False
Population weight still in features: False


Identify Feature Types

In [16]:
categorical_features = features.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_features = features.select_dtypes(
    include=[np.number]
).columns.tolist()

print("Categorical variables:", len(categorical_features))
print("Numerical variables:", len(numerical_features))
print("Total predictor variables:", len(features.columns))

Categorical variables: 417
Numerical variables: 26
Total predictor variables: 443


In [17]:
for column in categorical_features:
    features[column] = (
        features[column]
        .str.strip()
        .replace("", np.nan)
    )

print("Categorical values cleaned.")
print("The __NOT_ASKED__ category was preserved.")

Categorical values cleaned.
The __NOT_ASKED__ category was preserved.
